In [1]:
import pandas as pd
from tqdm import tqdm
from collections import Counter

In [13]:
def get_ahi_severity_class(ahi):
    if ahi < 5:     # Normal
        return 'normal'
    if ahi < 15:    # Mild
        return 'mild'
    if ahi < 30:    # Moderate
        return 'moderate'
    return 'severe'  # Severe

In [3]:
dataset_file_mesa = '/vol/sleepstudy/datasets/mesa/datasets/mesa-sleep-harmonized-dataset-0.7.0.csv'
dataset_file_cfs  = '/vol/sleepstudy/datasets/cfs/datasets/cfs-visit5-harmonized-dataset-0.7.0.csv'

demographic_columns = ['nsrr_age', 'nsrr_bmi', 'nsrr_sex']

ids_mesa = list(pd.read_csv('../wearsed/dataset/data_ids/mesa_ids_somnolyzer.csv')['id'])
ids_cfs  = list(pd.read_csv('../wearsed/dataset/data_ids/cfs_ids.csv', header=None)[0])

In [4]:
dataset_mesa = pd.read_csv(dataset_file_mesa)[['mesaid'] + demographic_columns].set_index('mesaid')
dataset_cfs  = pd.read_csv(dataset_file_cfs)[['nsrrid'] + demographic_columns].set_index('nsrrid')

dataset_mesa = dataset_mesa.join(pd.DataFrame({'mesaid': ids_mesa}).set_index('mesaid'), how='inner')
dataset_cfs  = dataset_cfs.join(pd.DataFrame({'nsrrid': ids_cfs}).set_index('nsrrid'), how='inner')

In [5]:
dataset_mesa.describe()

,nsrr_age,nsrr_bmi
count,1887.000000,1884.000000
mean,69.323794,28.746046
std,9.099461,5.573854
min,54.000000,16.290000
25%,62.000000,24.825000
50%,68.000000,27.980000
75%,76.000000,31.912500
max,90.000000,56.010000


In [6]:
dataset_cfs.describe()

,nsrr_age,nsrr_bmi
count,323.000000,323.000000
mean,39.489907,32.113826
std,19.650680,9.044692
min,7.290000,13.250461
25%,21.055000,25.737207
50%,42.110000,30.747440
75%,54.465000,38.105753
max,88.530000,61.957006


In [7]:
print('MESA:', Counter(dataset_mesa['nsrr_sex']), Counter(dataset_mesa['nsrr_sex'])['male'] / (Counter(dataset_mesa['nsrr_sex'])['female'] + Counter(dataset_mesa['nsrr_sex'])['male']), '% male')

print('CFS: ', Counter(dataset_cfs['nsrr_sex']), Counter(dataset_cfs['nsrr_sex'])['male'] / (Counter(dataset_cfs['nsrr_sex'])['female'] + Counter(dataset_cfs['nsrr_sex'])['male']), '% male')

MESA: Counter({'female': 1010, 'male': 877}) 0.4647588765235824 % male
CFS:  Counter({'female': 197, 'male': 126}) 0.39009287925696595 % male


### TSTs and AHIs

In [8]:
scorings_mesa = '/vol/sleepstudy/datasets/mesa/scorings/somnolyzer/'
scorings_cfs  = '/vol/sleepstudy/datasets/cfs/scorings/somnolyzer/' # TODO

In [9]:
tsts_mesa = []
tsts_cfs = []

evcount_mesa = []
evcount_cfs  = []

print('=== MESA')
print('- TST')
for id in tqdm(ids_mesa):
    tsts_mesa.append((pd.read_csv(scorings_mesa + f'hypnogram/hypnogram-{id:04}.csv', header=None)[0] > 0).sum() / (60*60))

print('- Event Count')
for id in tqdm(ids_mesa):
    ev_list = pd.read_csv(scorings_mesa + f'event_list/event-list-{id:04}.csv')
    evcount_mesa.append(len(ev_list[ev_list['Type'] != 'ARO']))

print('=== MESA')
print('- TST')
for id in tqdm(ids_cfs):
    tsts_cfs.append((pd.read_csv(scorings_cfs + f'hypnogram/hypnogram-{id}.csv', header=None)[0] > 0).sum() / (60*60))

print('- Event Count')
for id in tqdm(ids_cfs):
    ev_list = pd.read_csv(scorings_cfs + f'event_list/event-list-{id}.csv')
    evcount_cfs.append(len(ev_list[ev_list['Type'] != 'ARO']))

=== MESA
- TST


  0%|          | 0/1887 [00:00<?, ?it/s]

100%|██████████| 1887/1887 [00:31<00:00, 60.46it/s]


- Event Count


100%|██████████| 1887/1887 [00:23<00:00, 79.52it/s] 


=== MESA
- TST


100%|██████████| 323/323 [00:05<00:00, 53.98it/s]


- Event Count


100%|██████████| 323/323 [00:03<00:00, 87.06it/s] 


In [10]:
vals_mesa = pd.DataFrame({
    'TST': tsts_mesa,
    'Event Count': evcount_mesa
})
vals_mesa['AHI'] = vals_mesa['Event Count'] / vals_mesa['TST']

vals_cfs = pd.DataFrame({
    'TST': tsts_cfs,
    'Event Count': evcount_cfs
})
vals_cfs['AHI'] = vals_cfs['Event Count'] / vals_cfs['TST']

In [11]:
vals_mesa.describe()

,TST,Event Count,AHI
count,1887.000000,1887.00000,1887.000000
mean,6.222081,133.36248,21.867899
std,1.376625,112.02852,17.992980
min,0.650000,1.00000,0.278422
25%,5.408333,50.00000,8.334155
50%,6.300000,102.00000,16.540541
75%,7.150000,185.00000,30.762813
max,10.533333,850.00000,106.632911


In [12]:
vals_cfs.describe()

,TST,Event Count,AHI
count,323.000000,323.000000,323.000000
mean,6.372343,92.839009,15.203079
std,1.216075,115.508108,19.146634
min,2.066667,1.000000,0.154440
25%,5.737500,23.000000,3.521086
50%,6.441667,45.000000,6.978335
75%,7.062500,112.500000,19.729574
max,9.441667,784.000000,132.320675


In [16]:
sev_classes_mesa = {'normal': 0, 'mild': 1, 'moderate': 2, 'severe': 3}
sev_classes_cfs  = {'normal': 0, 'mild': 1, 'moderate': 2, 'severe': 3}

print('=== MESA')
for ahi in list(vals_mesa['AHI']):
    sev_class = get_ahi_severity_class(ahi)
    sev_classes_mesa[sev_class] += 1

print('=== CFS ===')
for ahi in list(vals_cfs['AHI']):
    sev_class = get_ahi_severity_class(ahi)
    sev_classes_cfs[sev_class] += 1

=== MESA
=== CFS ===


In [17]:
print(sev_classes_mesa)
print(sev_classes_cfs)

{'normal': 247, 'mild': 606, 'moderate': 552, 'severe': 488}
{'normal': 121, 'mild': 105, 'moderate': 50, 'severe': 53}
